In [2]:
import pickle
import numpy as np
import requests
from difflib import get_close_matches
from sklearn.metrics.pairwise import cosine_similarity
import ast
import nltk
from nltk.stem import PorterStemmer

In [8]:
new_df     = pickle.load(open('new_df.pkl', 'rb'))
similarity = pickle.load(open('similarity.pkl', 'rb'))
cv         = pickle.load(open('cv.pkl', 'rb'))
vector     = pickle.load(open('vector.pkl', 'rb'))

TMDB_API_KEY = "6db63c02f7b0456c849b9712eb682ec2"
ps = PorterStemmer()

In [4]:
def stems(text):
    return " ".join([ps.stem(i) for i in text.split()])

def fetch_movie_data_from_api(movie_title):
    # Step 1: search for the movie
    search_url = f"https://api.themoviedb.org/3/search/movie"
    search_res = requests.get(search_url, params={
        "api_key": TMDB_API_KEY,
        "query": movie_title
    }).json()

    if not search_res['results']:
        return None

    movie = search_res['results'][0]
    movie_id = movie['id']
    overview = movie.get('overview', '')

    # Step 2: fetch full details (genres, keywords, credits)
    details_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    details = requests.get(details_url, params={"api_key": TMDB_API_KEY}).json()

    keywords_url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
    keywords = requests.get(keywords_url, params={"api_key": TMDB_API_KEY}).json()

    credits_url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    credits = requests.get(credits_url, params={"api_key": TMDB_API_KEY}).json()

    # Step 3: extract fields same way as original pipeline
    genres   = [g['name'].replace(" ", "") for g in details.get('genres', [])]
    kws      = [k['name'].replace(" ", "") for k in keywords.get('keywords', [])]
    cast     = [c['name'].replace(" ", "") for c in credits.get('cast', [])[:4]]
    director = [
        m['name'].replace(" ", "")
        for m in credits.get('crew', [])
        if m['job'] == 'Director'
    ][:1]
    overview_words = overview.split()

    tags = " ".join(genres + kws + overview_words + cast + director).lower()
    tags = stems(tags)

    return {
        "title": movie.get('title', movie_title),
        "tags": tags
    }

In [6]:
def recommend(movie_title):
    titles = new_df['title'].tolist()

    # --- local fuzzy match first ---
    matches = get_close_matches(movie_title, titles, n=1, cutoff=0.6)

    if matches:
        matched_title = matches[0]
        if matched_title.lower() != movie_title.lower():
            print(f"Showing results for: '{matched_title}' (local)\n")

        idx = new_df.index.get_loc(new_df[new_df['title'] == matched_title].index[0])
        distance = sorted(enumerate(similarity[idx]), reverse=True, key=lambda x: x[1])
        print("Recommended movies:")
        for i in distance[1:6]:
            print(" -", new_df.iloc[i[0]].title)

    # --- API fallback ---
    else:
        print(f"'{movie_title}' not found locally, fetching from TMDB...\n")
        data = fetch_movie_data_from_api(movie_title)

        if not data:
            print("Couldn't find this movie on TMDB either.")
            return

        print(f"Showing results for: '{data['title']}' (via TMDB API)\n")

        # vectorize the API movie using the SAME fitted cv
        api_vector = cv.transform([data['tags']]).toarray()

        # compare against all local movie vectors
        api_similarity = cosine_similarity(api_vector, vector)[0]

        distance = sorted(enumerate(api_similarity), reverse=True, key=lambda x: x[1])
        print("Recommended movies:")
        for i in distance[:5]:
            print(" -", new_df.iloc[i[0]].title)

In [7]:
# Should hit local data
recommend('Avengers')

Showing results for: 'The Avengers' (local)

Recommended movies:
 - Avengers: Age of Ultron
 - Captain America: Civil War
 - X-Men: Apocalypse
 - X-Men
 - Ant-Man


In [9]:
# Should fall through to TMDB API (if not in your dataset)
recommend('Oppenheimer')

'Oppenheimer' not found locally, fetching from TMDB...

Showing results for: 'Oppenheimer' (via TMDB API)

Recommended movies:
 - Red Tails
 - Schindler's List
 - The Imitation Game
 - Saints and Soldiers
 - Black Book
